<a href="https://colab.research.google.com/github/rm571222/dataholics-oracle-challenge/blob/main/notebooks/02_data_upload/nb6_upload_hospital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB6 — Carga no Oracle: Dimensão Hospital (T_SIH_HOSPITAL)

**Projeto DATAHOLICS — FIAP Challenge | Parceria Oracle**

Este notebook documenta a carga da dimensão de hospitais — identificação, localização e capacidade instalada de leitos — no Oracle, com base nas decisões da fase de exploração (NB2).

## Estrutura deste notebook
1. Bibliotecas e autenticação
2. Criação da tabela (DDL)
3. Extração dos hospitais únicos e da capacidade de leitos
4. Carga e criação da chave estrangeira com a tabela fato

In [ ]:
!pip install -q oracledb requests pandas

from google.colab import files
uploaded = files.upload()  # selecione o Wallet_SPRINT02CHALLENGE.zip

import zipfile, os
wallet_path = '/content/wallet'
os.makedirs(wallet_path, exist_ok=True)
with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
    z.extractall(wallet_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.1 MB/s eta 0:00:00


Saving Wallet_SPRINT02CHALLENGE.zip to Wallet_SPRINT02CHALLENGE.zip


In [ ]:
import oracledb, pandas as pd, math, requests, zipfile as zf, io
import getpass

senha_banco = getpass.getpass("Senha do banco: ")
senha_wallet = getpass.getpass("Senha do wallet: ")

conn = oracledb.connect(
    user="ADMIN",
    password=senha_banco,
    dsn="sprint02challenge_high",
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=senha_wallet
)
cursor = conn.cursor()
cursor.execute("ALTER SESSION DISABLE PARALLEL DML")
cursor.execute("ALTER SESSION DISABLE PARALLEL QUERY")

def inserir_em_lotes(cursor, conn, sql, dados, tamanho_lote=5000):
    total_ignorados = 0
    for i in range(0, len(dados), tamanho_lote):
        lote = dados[i:i + tamanho_lote]
        cursor.executemany(sql, lote, batcherrors=True)
        erros = cursor.getbatcherrors()
        if erros:
            total_ignorados += len(erros)
        conn.commit()
    if total_ignorados > 0:
        print(f'  ({total_ignorados} linhas ignoradas por chave primária duplicada)')
    return total_ignorados

def normalizar_tupla(row):
    resultado = []
    for x in row:
        if hasattr(x, 'item'):
            x = x.item()
        if isinstance(x, float) and math.isnan(x):
            x = None
        resultado.append(x)
    return tuple(resultado)

print("Conectado com sucesso!")

Senha do banco: ··········
Senha do wallet: ··········
Conectado com sucesso!


## 2. Criação da tabela (DDL)

A dimensão hospital guarda apenas identificação, localização e capacidade de leitos — sem o campo `cd_tipo_gestao`, que foi avaliado e descartado na exploração por representar um atributo de transação (variável por internação), não uma característica fixa do hospital.

In [ ]:
cursor.execute("""
    BEGIN
        EXECUTE IMMEDIATE 'DROP TABLE T_SIH_HOSPITAL';
    EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
    END;
""")

cursor.execute("""
    CREATE TABLE T_SIH_HOSPITAL (
        cd_hospital              VARCHAR2(10),
        cd_municipio             VARCHAR2(6),
        qt_leitos_existentes     NUMBER(5),
        qt_leitos_sus            NUMBER(5),
        qt_leitos_uti_total      NUMBER(5),
        qt_leitos_uti_sus        NUMBER(5),
        qt_leitos_uti_adulto     NUMBER(5),
        qt_leitos_uti_pediatrico NUMBER(5),
        qt_leitos_uti_neonatal   NUMBER(5),
        CONSTRAINT PK_SIH_HOSPITAL PRIMARY KEY (cd_hospital)
    )
""")
conn.commit()
print("Tabela T_SIH_HOSPITAL criada")

## 3. Extração dos hospitais únicos e da capacidade de leitos

Os hospitais vêm da própria `T_SIH_INTERNACAO` (já carregada no NB5) — só entram na dimensão os estabelecimentos que efetivamente geraram internação. A capacidade de leitos vem do CSV oficial de Leitos SUS, usando a **última competência disponível por hospital individualmente** (decisão validada na exploração, NB2) para não perder hospitais que saíram do cadastro antes da competência mais recente.

In [ ]:
cursor.execute("SELECT DISTINCT cd_hospital, cd_municipio_hospital FROM T_SIH_INTERNACAO")
colunas_result = [c[0].lower() for c in cursor.description]
df_dimensao_hospital = pd.DataFrame(cursor.fetchall(), columns=colunas_result)
df_dimensao_hospital = df_dimensao_hospital.rename(columns={'cd_municipio_hospital': 'cd_municipio'})
df_dimensao_hospital['cd_hospital'] = df_dimensao_hospital['cd_hospital'].astype(str)

print(f"Hospitais únicos na dimensão: {len(df_dimensao_hospital)}")

In [ ]:
resp = requests.get("https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/Leitos_SUS/Leitos_csv_2026.zip")
z = zf.ZipFile(io.BytesIO(resp.content))
with z.open(z.namelist()[0]) as f:
    df_leitos_raw = pd.read_csv(f, sep=';', encoding='latin1')

df_leitos_sp = (
    df_leitos_raw[df_leitos_raw['UF'] == 'SP']
    .sort_values('COMP')
    .drop_duplicates(subset='CNES', keep='last')
    .copy()
)

colunas_leitos = ['CNES', 'LEITOS_EXISTENTES', 'LEITOS_SUS',
                   'UTI_TOTAL_EXIST', 'UTI_TOTAL_SUS',
                   'UTI_ADULTO_EXIST', 'UTI_PEDIATRICO_EXIST', 'UTI_NEONATAL_EXIST']
df_leitos_sp = df_leitos_sp[colunas_leitos].copy()
df_leitos_sp['CNES'] = df_leitos_sp['CNES'].astype(str)
df_leitos_sp = df_leitos_sp.rename(columns={
    'CNES': 'cd_hospital', 'LEITOS_EXISTENTES': 'qt_leitos_existentes', 'LEITOS_SUS': 'qt_leitos_sus',
    'UTI_TOTAL_EXIST': 'qt_leitos_uti_total', 'UTI_TOTAL_SUS': 'qt_leitos_uti_sus',
    'UTI_ADULTO_EXIST': 'qt_leitos_uti_adulto', 'UTI_PEDIATRICO_EXIST': 'qt_leitos_uti_pediatrico',
    'UTI_NEONATAL_EXIST': 'qt_leitos_uti_neonatal'
})

print(f"Hospitais no cadastro de leitos (SP, última competência por hospital): {len(df_leitos_sp)}")

## 4. Carga e criação da chave estrangeira

O join é `LEFT` — todo hospital da tabela fato entra na dimensão, com leitos nulos onde não houver correspondência. A FK com `T_SIH_INTERNACAO` só é criada **depois** da carga, já que o Oracle exige que a tabela referenciada esteja populada antes de validar a constraint.

In [ ]:
tabela2_final = df_dimensao_hospital.merge(df_leitos_sp, on='cd_hospital', how='left')

colunas_numericas_t2 = ['qt_leitos_existentes', 'qt_leitos_sus', 'qt_leitos_uti_total',
                         'qt_leitos_uti_sus', 'qt_leitos_uti_adulto',
                         'qt_leitos_uti_pediatrico', 'qt_leitos_uti_neonatal']
for col in colunas_numericas_t2:
    tabela2_final[col] = pd.to_numeric(tabela2_final[col], errors='coerce')

print(f"Hospitais na dimensão: {len(df_dimensao_hospital)}")
print(f"Com leitos casados: {tabela2_final['qt_leitos_existentes'].notna().sum()}")

COLUNAS_T2 = ['cd_hospital', 'cd_municipio'] + colunas_numericas_t2
INSERT_SQL_T2 = f"INSERT INTO T_SIH_HOSPITAL ({','.join(COLUNAS_T2)}) VALUES ({','.join([':'+str(i+1) for i in range(len(COLUNAS_T2))])})"

dados_t2 = [normalizar_tupla(row) for row in tabela2_final[COLUNAS_T2].itertuples(index=False, name=None)]
inserir_em_lotes(cursor, conn, INSERT_SQL_T2, dados_t2)

cursor.execute("SELECT COUNT(*) FROM T_SIH_HOSPITAL")
print(f"\nTotal na T_SIH_HOSPITAL: {cursor.fetchone()[0]}")

In [ ]:
cursor.execute("""
    ALTER TABLE T_SIH_INTERNACAO
        ADD CONSTRAINT FK_SIH_INTERNACAO_HOSPITAL
        FOREIGN KEY (cd_hospital) REFERENCES T_SIH_HOSPITAL(cd_hospital)
""")
conn.commit()
print("FK_SIH_INTERNACAO_HOSPITAL criada com sucesso")

## Conclusão — NB6

A tabela `T_SIH_HOSPITAL` foi carregada com 650 hospitais, 86,8% com correspondência de leitos (564 de 650) — cobertura elevada em relação à estratégia inicial ao usar a última competência disponível por hospital. O residual de 13,2% concentra-se em estabelecimentos que, por natureza, não operam com leito de internação formal (Hospital-Dia, AME, UPA, Pronto-Socorro), conforme investigado na exploração (NB2). A FK com a tabela fato está ativa e validada.